In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression

# ---------------- LOAD DATA ----------------

data = pd.read_csv("DATASET.csv")

data = data.drop(columns=["Unnamed: 0","MATCH_ID"], errors="ignore")

# =--------------=----------=-=-=-ensure categorical type
cat_cols = ["PLAYER_NAME","OPPONENT_TEAM","VENUE","MATCH_FORMAT"]

for c in cat_cols:
    data[c] = data[c].astype(str)

# ---------------- TARGET VARIABLES ----------------

y = data[[
    "RUNS_SCORED",
    "BALLS_FACED",
    "FOURS",
    "SIXES"
]]

# ----------------             FEATURES ----------------

X = data[cat_cols]

# ---------------- ENCODER ----------------

encoder = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ],
    remainder="drop"
)

X_encoded = encoder.fit_transform(X)

# ---------------- TRAIN TEST SPLIT ----------------

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

# ---------------- MODEL ----------------

model = LinearRegression()

model.fit(X_train, y_train)



# ---------------- USER INPUT ----------------

player = input("Enter Player: ").upper()
opponent = input("Enter Opponent Team: ").upper()
venue = input("Enter Venue: ").upper()
format_match = input("Enter Format (only TEST/ODI): ").upper()

user = pd.DataFrame({
    "PLAYER_NAME":[player],
    "OPPONENT_TEAM":[opponent],
    "VENUE":[venue],
    "MATCH_FORMAT":[format_match]
})

user_encoded = encoder.transform(user)

# ---------------- PREDICTION ----------------

prediction = model.predict(user_encoded)[0]

runs = max(0, prediction[0])
balls = max(1, prediction[1])
fours = max(0, prediction[2])
sixes = max(0, prediction[3])

# derived metrics
strike_rate = (runs / balls) * 100
boundary_rate = (fours + sixes) / balls



print("Player name:-",player)
print("Opponent Team:-",opponent)
print("Match Venue / City/ Ground:-",venue)
print("Match Format:-",format_match)

print("\nPredicted Performance")
print("----------------------")

if balls>=100 and runs>= 100:
    print(player,"will score above 100+")
else:
    print("Runs:-", round(runs))
    print("Balls:-", round(balls))

if runs>=80:
    print("    Agressive Playing⚡⚡🔥")   

print("Fours:-", round(fours))
print("Sixes:-", round(sixes))
print("Strike Rate:-", round(strike_rate,2)) 
print("Boundary per ball:-", round(boundary_rate,3))

Player name:- VIRAT KOHLI
Opponent Team:- ENGLAND
Match Venue / City/ Ground:- DELHI
Match Format:- T20

Predicted Performance
----------------------
VIRAT KOHLI will score above 100+
Agressive Playing⚡⚡🔥
Fours:- 20
Sixes:- 12
Strike Rate:- 136.3
Boundary per ball:- 0.184
